# Notebook 03: Feature engineering

**Project:** Lightweight machine-learning-based cybersecurity threat detection for resource-constrained hardware
**Author:** Anush Jindal

Groups the 78 features, decides whether to engineer new ones, and defines the four nested
feature lists used in every later experiment: **A (58), B (39), C (15), D (6)**. No model
is trained here; notebook 04 trains a tree on each list.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')
%matplotlib inline
print("Libraries imported.")

Mounted at /content/drive
Libraries imported.


## Step 1 — Load the split from notebook 02

The four files saved at the end of `02_preprocessing.ipynb`.

In [ ]:
X_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/threat detector/data/X_train.csv")
X_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/threat detector/data/X_test.csv")
y_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/threat detector/data/y_train.csv")
y_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/threat detector/data/y_test.csv")

# Should match notebook 02: (488393, 78) and (122099, 78)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(488393, 78)
(122099, 78)
(488393, 1)
(122099, 1)


## Step 2 — Group the features

Every column of `X_train` sorted into five groups: traffic volume, connection behaviour
(durations and inter-arrival times), packet characteristics, protocol (flags, TCP window,
header lengths) and port. The groups are used in Step 4 to pick one or two
representatives per behaviour for the smaller lists.

In [ ]:
feature_groups = {
    "traffic_volume": [
        "Total Fwd Packets",
        "Total Backward Packets",
        "Total Length of Fwd Packets",
        "Total Length of Bwd Packets",
        "Flow Bytes/s",
        "Flow Packets/s",
        "Fwd Packets/s",
        "Bwd Packets/s",
        "Down/Up Ratio",
        "act_data_pkt_fwd",
        "Subflow Fwd Packets",
        "Subflow Fwd Bytes",
        "Subflow Bwd Packets",
        "Subflow Bwd Bytes",
        "Fwd Avg Bytes/Bulk",
        "Fwd Avg Packets/Bulk",
        "Fwd Avg Bulk Rate",
        "Bwd Avg Bytes/Bulk",
        "Bwd Avg Packets/Bulk",
        "Bwd Avg Bulk Rate",
    ],
    "connection_behavior": [
        "Flow Duration",
        "Flow IAT Mean",
        "Flow IAT Std",
        "Flow IAT Max",
        "Flow IAT Min",
        "Fwd IAT Total",
        "Fwd IAT Mean",
        "Fwd IAT Std",
        "Fwd IAT Max",
        "Fwd IAT Min",
        "Bwd IAT Total",
        "Bwd IAT Mean",
        "Bwd IAT Std",
        "Bwd IAT Max",
        "Bwd IAT Min",
        "Active Mean",
        "Active Std",
        "Active Max",
        "Active Min",
        "Idle Mean",
        "Idle Std",
        "Idle Max",
        "Idle Min",
    ],
    "packet_characteristics": [
        "Fwd Packet Length Max",
        "Fwd Packet Length Min",
        "Fwd Packet Length Mean",
        "Fwd Packet Length Std",
        "Bwd Packet Length Max",
        "Bwd Packet Length Min",
        "Bwd Packet Length Mean",
        "Bwd Packet Length Std",
        "Min Packet Length",
        "Max Packet Length",
        "Packet Length Mean",
        "Packet Length Std",
        "Packet Length Variance",
        "Average Packet Size",
        "Avg Fwd Segment Size",
        "Avg Bwd Segment Size",
    ],
    "protocol": [
        "Fwd PSH Flags",
        "Bwd PSH Flags",
        "Fwd URG Flags",
        "Bwd URG Flags",
        "FIN Flag Count",
        "SYN Flag Count",
        "RST Flag Count",
        "PSH Flag Count",
        "ACK Flag Count",
        "URG Flag Count",
        "CWE Flag Count",
        "ECE Flag Count",
        "Init_Win_bytes_forward",
        "Init_Win_bytes_backward",
        "min_seg_size_forward",
        "Fwd Header Length",
        "Bwd Header Length",
        "Fwd Header Length.1",
    ],
    "port_info": [
        "Destination Port",
    ],
}

## Step 3 — Engineered features

A new column would have to be computed from each row's own values only, and added
identically to `X_train` and `X_test` (never from a statistic computed across the
dataset).

In [ ]:
# No engineered features are added:
#  - a decision tree split depends only on the order of a feature's values, so monotonic
#    transforms (log, scaling) change nothing;
#  - the CICFlowMeter features already include the obvious ratios and rates
#    (Down/Up Ratio, Flow Bytes/s, Flow Packets/s, Fwd/Bwd Packets/s);
#  - every extra column adds memory and work per flow on the target hardware.
# The *_engineered.csv files saved in Step 6 are therefore identical to X_train / X_test.

## Step 4 — Feature lists A, B, C, D

Four nested lists, from large to very small (D ⊂ C ⊂ B ⊂ A). Each removal reason for list
A was tested on the data in notebook 04, Step 3.

In [ ]:
# Names are the stripped versions (after df.columns.str.strip()).
# Method: domain reasoning (see research/feature_documentation.md).

all_features = [f for group in feature_groups.values() for f in group]

# --- Experiment A: large set (58 features) ---
# A = all 78 features minus the 20 columns dropped below. Every drop reason was
# tested on the real data in notebook 4, Step 3 (results in experiment_notes.md):
#   - always zero in this file (1 distinct value): Bwd PSH Flags, Fwd URG Flags,
#     Bwd URG Flags, CWE Flag Count and the 6 Bulk columns
#   - exact copies of a kept column: Avg Fwd/Bwd Segment Size, Subflow Fwd Packets,
#     Subflow Fwd Bytes, Subflow Bwd Packets, Fwd Header Length.1
#   - same ordering as a kept column (a tree only uses ordering): Packet Length Variance
#   - NEARLY identical to a kept column (decided 23-10-2026 to keep them out):
#       Subflow Bwd Bytes   vs Total Length of Bwd Packets: differs in 34 of 488,393 rows
#       ECE Flag Count      vs RST Flag Count:              differs in 1 row
#       Average Packet Size vs Packet Length Mean:          Spearman 0.9976
# Fwd PSH Flags USED to be dropped as "always zero", but it is non-zero in 4.32% of
# training rows, so it carries real information. It was added back to A on 23-10-2026
# (A went from 57 to 58 features).
A_drop = [
    "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "CWE Flag Count", "ECE Flag Count", "Packet Length Variance",
    "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
]
experiment_A_features = [f for f in all_features if f not in A_drop]

# --- Experiment B: roughly half of all 78 features (39) ---
# A minus 19 secondary variants: extra IAT/active/idle statistics, min-length
# features, header lengths, URG count, and Fwd PSH Flags (B already has
# PSH Flag Count, which counts PSH flags in both directions).
# Keeping Fwd PSH Flags out of B means B, C and D are exactly the same lists as
# before 23-10-2026, so only the runs built on A have to be redone in notebook 4.
B_drop = [
    "Fwd IAT Total", "Fwd IAT Std", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Min",
    "Active Std", "Active Max", "Active Min",
    "Idle Std", "Idle Min",
    "Fwd Packet Length Min", "Bwd Packet Length Min", "Min Packet Length",
    "URG Flag Count", "Fwd Header Length", "Bwd Header Length",
    "Fwd PSH Flags",
]
experiment_B_features = [f for f in experiment_A_features if f not in B_drop]

# --- Experiment C: most important by judgment (15) ---
# One or two strong representatives per behaviour: volume, rate, size, timing,
# handshake, TCP window and port.
experiment_C_features = [
    "Destination Port",
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Fwd Packet Length Max",
    "Packet Length Mean",
    "Packet Length Std",
    "Flow IAT Mean",
    "Flow IAT Std",
    "SYN Flag Count",
    "Init_Win_bytes_forward",
    "Idle Mean",
]

# --- Experiment D: very small set (6) ---
experiment_D_features = [
    "Destination Port",
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets",
    "Init_Win_bytes_forward",
]

# Sanity checks: sets are nested (D ⊂ C ⊂ B ⊂ A) and every name exists in X_train
sets = {"A": experiment_A_features, "B": experiment_B_features,
        "C": experiment_C_features, "D": experiment_D_features}
assert set(experiment_D_features) <= set(experiment_C_features) <= set(experiment_B_features) <= set(experiment_A_features)
for name, feats in sets.items():
    missing = [f for f in feats if f not in X_train.columns]
    print(name, len(feats), "missing:", missing)

# The sizes notebook 4 expects (Step 2 checks these again after loading)
EXPECTED_SIZES = {"A": 58, "B": 39, "C": 15, "D": 6}
for name, n in EXPECTED_SIZES.items():
    assert len(sets[name]) == n, f"List {name} has {len(sets[name])} features, expected {n}"
assert "Fwd PSH Flags" in experiment_A_features and "Fwd PSH Flags" not in experiment_B_features
print("Sizes match: A 58, B 39, C 15, D 6")

## Step 5 — Feature documentation

The reason for keeping or removing each of the 78 features, and which lists it is in, is
recorded in `research/feature_documentation.md`.

## Step 6 — Save

The engineered `X_train` / `X_test` and the four lists are saved for notebook 04. The
same `feature_sets.json` is kept in the repository as `experiments/feature_sets.json`.

In [ ]:
import json
from pathlib import Path

# Define output directory using Path
output_dir = Path("/content/drive/MyDrive/Colab Notebooks/threat detector/data")
output_dir.mkdir(parents=True, exist_ok=True)

# Save engineered datasets
X_train.to_csv(output_dir / "X_train_engineered.csv", index=False)
X_test.to_csv(output_dir / "X_test_engineered.csv", index=False)

# Store our feature lists in a dictionary
feature_sets = {
    "A": experiment_A_features,
    "B": experiment_B_features,
    "C": experiment_C_features,
    "D": experiment_D_features
}

# Save feature sets dictionary to a JSON file
with open(output_dir / "feature_sets.json", "w") as f:
    json.dump(feature_sets, f, indent=4)

print("Engineered datasets and feature sets successfully saved to:", output_dir)

Engineered datasets and feature sets successfully saved to: /content/drive/MyDrive/Colab Notebooks/threat detector/data
